In [2]:
pip install numpy pandas matplotlib statsmodels scikit-learn pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 25.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 44.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 47.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 46.4 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 45.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 46.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 52.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 45.5 MB/s  0:00:006m0:00:01
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/17 [numpy]  WARNING: The scripts f2py and numpy-config are installed in '/usr/local/pyth

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Load your data
loandata = pd.read_csv("Climate Data/BUSLOANS.csv")
unemployment = pd.read_csv("Climate Data/UNEMPLOYMENT.csv")
cpi = pd.read_csv("Climate Data/CPI.csv")

# Convert dates to datetime
loandata['Date'] = pd.to_datetime(loandata['observation_date'])
unemployment['Date'] = pd.to_datetime(unemployment['observation_date'])
cpi['Date'] = pd.to_datetime(cpi['observation_date'])

# Merge datasets
merged_data = pd.merge(loandata, unemployment, on='observation_date', how='inner')
merged_data = pd.merge(merged_data, cpi, on='observation_date', how='inner')

# Calculate loan growth and inflation
merged_data['loan_growth'] = merged_data['BUSLOANS'].pct_change() * 100
merged_data['inflation'] = merged_data['CPIAUCSL'].pct_change() * 100

# Drop NaN values
merged_data = merged_data.dropna()

# Set date as index
merged_data['observation_date'] = pd.to_datetime(merged_data['observation_date'])
merged_data.set_index('observation_date', inplace=True)

loan_growth = merged_data['loan_growth']
unemployment_rate = merged_data['UNRATE']

# Grid search - with better error reporting
best_aic = np.inf
best_params = None
results_list = []

for p, d, q in product(range(0, 5), range(0, 2), range(0, 5)):
    try:
        model = sm.tsa.SARIMAX(
            loan_growth, 
            order=(p, d, q),
            exog=unemployment_rate
        )
        results = model.fit(disp=False)
        results_list.append({
            'order': f'({p},{d},{q})', 
            'AIC': results.aic, 
            'BIC': results.bic
        })
        if results.aic < best_aic:
            best_aic = results.aic
            best_params = (p, d, q)
    except Exception as e:
        print(f"Failed for order ({p},{d},{q}): {str(e)[:50]}")
        continue

# Display results
if results_list:
    results_df = pd.DataFrame(results_list).sort_values('AIC')
    print(results_df.head(10))
    print(f"\nOptimal ARIMA order: {best_params}")
else:
    print("No successful model fits!")

/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored

      order          AIC          BIC
33  (3,0,3)  2203.948480  2242.664292
24  (2,0,4)  2205.154260  2243.870072
34  (3,0,4)  2205.663231  2249.218519
43  (4,0,3)  2205.727294  2249.282582
44  (4,0,4)  2207.677750  2256.072514
12  (1,0,2)  2209.159377  2233.356759
13  (1,0,3)  2209.544916  2238.581775
22  (2,0,2)  2209.735159  2238.772018
31  (3,0,1)  2209.983391  2239.020250
32  (3,0,2)  2210.844210  2244.720545

Optimal ARIMA order: (3, 0, 3)


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Load your data directly from CSV files with correct paths
def clean_fred_data(file_path, value_col_name):
    df = pd.read_csv(file_path)
    date_col = [c for c in df.columns if 'date' in c.lower()][0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col)
    df_numeric = df.apply(pd.to_numeric, errors='coerce')
    df_q = df_numeric.resample('QE').mean()
    df_q.columns = [value_col_name]
    return df_q

# Load data with correct paths to Climate Data directory
df_loans = clean_fred_data('Climate Data/BUSLOANS.csv', 'Loans_Billions')
df_cpi = clean_fred_data('Climate Data/CPI.csv', 'CPI')
df_unemp = clean_fred_data('Climate Data/UNEMPLOYMENT.csv', 'Unemployment_Rate')
df_charge = clean_fred_data('Climate Data/CORBLACBS.csv', 'Chargeoff_Rate')
df_gdp = clean_fred_data('Climate Data/GDPC1.csv', 'GDP')

df_history = pd.concat([df_loans, df_cpi, df_unemp, df_charge, df_gdp], axis=1).dropna()

# Create calculated columns
df_history['Loan_Growth'] = df_history['Loans_Billions'].pct_change(4) * 100
df_history['Inflation'] = df_history['CPI'].pct_change(4) * 100
df_history['GDP_Growth'] = df_history['GDP'].pct_change(4) * 100
df_history = df_history.dropna()

# Create COVID dummy
covid_start = '2020-03-31'
covid_end = '2021-12-31'
df_history['COVID_Dummy'] = 0
df_history.loc[(df_history.index >= covid_start) & (df_history.index <= covid_end), 'COVID_Dummy'] = 1

# Now run the ARIMAX grid search
exog_vars = ['GDP_Growth', 'Unemployment_Rate', 'Inflation']
y = df_history['Chargeoff_Rate']
X = df_history[exog_vars]

p_range = range(0, 4)
d_range = range(0, 3)
q_range = range(0, 4)

results_df = []

print("=" * 80)
print("ARIMAX Grid Search - Finding Best Specifications for Chargeoff_Rate")
print("=" * 80)
print(f"\nSearching {len(list(p_range)) * len(list(d_range)) * len(list(q_range))} combinations...")
print("This may take a few moments...\n")

counter = 0
for p in p_range:
    for d in d_range:
        for q in q_range:
            try:
                counter += 1
                model = ARIMA(y, order=(p, d, q), exog=X)
                fitted_model = model.fit()
               
                results_df.append({
                    'p': p,
                    'd': d,
                    'q': q,
                    'AIC': fitted_model.aic,
                    'BIC': fitted_model.bic,
                    'RMSE': np.sqrt(fitted_model.mse),
                    'LogLikelihood': fitted_model.llf
                })
               
                if counter % 10 == 0:
                    print(f"  Completed {counter} models...")
            except Exception as e:
                continue

results_table = pd.DataFrame(results_df)
results_table = results_table.sort_values('AIC').reset_index(drop=True)

print("\n" + "=" * 80)
print("TOP 10 MODELS BY AIC")
print("=" * 80)
print(results_table.head(10).to_string(index=False))

print("\n" + "=" * 80)
print("TOP 10 MODELS BY BIC")
print("=" * 80)
print(results_table.sort_values('BIC').head(10).to_string(index=False))

best_model_params = results_table.iloc[0]
print("\n" + "=" * 80)
print("BEST MODEL SPECIFICATION (by AIC)")
print("=" * 80)
print(f"ARIMAX{(int(best_model_params['p']), int(best_model_params['d']), int(best_model_params['q']))}")
print(f"AIC: {best_model_params['AIC']:.2f}")
print(f"BIC: {best_model_params['BIC']:.2f}")
print(f"RMSE: {best_model_params['RMSE']:.6f}")
print("=" * 80)

best_p, best_d, best_q = int(best_model_params['p']), int(best_model_params['d']), int(best_model_params['q'])
best_arimax = ARIMA(y, order=(best_p, best_d, best_q), exog=X).fit()

print("\nBEST MODEL SUMMARY")
print("=" * 80)
print(best_arimax.summary())

ARIMAX Grid Search - Finding Best Specifications for Chargeoff_Rate

Searching 48 combinations...
This may take a few moments...

  Completed 10 models...
  Completed 20 models...
  Completed 30 models...
  Completed 40 models...

TOP 10 MODELS BY AIC
 p  d  q         AIC         BIC     RMSE  LogLikelihood
 2  0  2 -135.957184 -108.337046 0.158014      76.978592
 3  0  1 -135.453459 -107.833321 0.158793      76.726730
 1  0  3 -134.834563 -107.214425 0.157475      76.417282
 1  1  2 -134.259264 -112.821099 0.198892      74.129632
 0  1  3 -134.076679 -112.638514 0.199003      74.038340
 3  0  2 -133.419578 -102.730536 0.156113      76.709789
 0  1  2 -133.309838 -114.934268 0.198907      72.654919
 1  0  2 -132.932828 -108.381594 0.158942      74.466414
 1  1  3 -132.343250 -107.842489 0.198716      74.171625
 2  1  2 -132.262616 -107.761855 0.198871      74.131308

TOP 10 MODELS BY BIC
 p  d  q         AIC         BIC     RMSE  LogLikelihood
 0  1  2 -133.309838 -114.934268 0.198907 